# OSNet: целевые проверки после аудита

Новый изолированный эксперимент. MVP, старые результаты и разметка организаторов не меняются.
По умолчанию запускается только **pilot**: один seed, пять условий, до 8 500 updates.
Это B0, прежняя и исправленная color-ветви, разрешение 256 и triplet. NiVe не используется.

**Restart Kernel → Run All** в прежнем `.venv`. Прогресс: условие, seed, шаг/бюджет, время и ETA.
Не запускай одновременно другой training notebook. Подробности и ограничения — в `README.md`.


In [1]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd() / 'Car-classification-MSK', *Path.cwd().parents]
REPO = next((p for p in candidates if (p / 'training/osnet_review_protocol.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Открой notebook из проекта Car-classification-MSK')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import torch
from IPython.display import Markdown, display
from training.osnet_ablation_suite import Budget
from training.osnet_review_protocol import prepare, run

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)
print('Repository:', REPO, '| Torch:', torch.__version__)


Repository: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK | Torch: 2.14.0


## 1. Заранее фиксируем правила

`pilot` — только первый seed, без выбора победителя и outer-оценки.
После разбора пилота переключи только `PHASE = 'confirm_inner'`: ещё два primary seed,
выбор одного общего шага на рецепт, затем alternate на трёх seed строго на этом шаге.
Alternate не меняет победителя или длительность. Draws — не независимые training seed.

`final` — отдельный запуск после разбора confirmation; требует `ALLOW_OUTER_EVALUATION = True`.
Все final веса фиксируются до первой outer-оценки. Outer ранее использовалась для выбора MVP:
она остаётся development validation, а не новым независимым тестом.


In [2]:
RUN_NAME = 'review_v1'
PHASE = 'final'  # 'pilot' -> 'confirm_inner' -> 'final', вручную после анализа
ALLOW_OUTER_EVALUATION = True
ALLOW_CPU_TRAINING = False
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
BUDGET = Budget(max_steps=1700, evaluation_interval=200, warmup_steps=100)
SEEDS = (20260915, 20260916, 20260917)
DRAW_SEEDS = (20260915, 20261016, 20261117)

print('Run:', RUN_NAME, '| phase:', PHASE, '| device:', DEVICE)
print('Training seeds:', SEEDS, '| fixed draws:', DRAW_SEEDS)
print('Pilot: 5 stages /', 5 * BUDGET.max_steps, 'updates')
print('Inner confirmation including pilot: at most 30 stages /', 30 * BUDGET.max_steps, 'updates')
print('No NiVe; no automatic next phase; no MVP replacement.')


Run: review_v1 | phase: final | device: mps
Training seeds: (20260915, 20260916, 20260917) | fixed draws: (20260915, 20261016, 20261117)
Pilot: 5 stages / 8500 updates
Inner confirmation including pilot: at most 30 stages / 51000 updates
No NiVe; no automatic next phase; no MVP replacement.


## 2. Проверяем входы — обучение ещё не начинается

Хеши изображений, CSV, splits, исходников, stock/MVP и масок записываются в новый manifest.
При несовпадении существующего manifest не удаляй его и не обходи проверку.
Изменение кода, seed, budgets или runtime требует нового RUN_NAME; переключать PHASE можно.


In [3]:
context = prepare(RUN_NAME, device=DEVICE, budget=BUDGET, seeds=SEEDS, draw_seeds=DRAW_SEEDS)
print('Output:', context['output'])
print('Fingerprint:', context['signature'])
for fold, partition in context['manifest']['inner'].items():
    print(fold, '| train IDs:', len(partition['train']), '| validation IDs:', len(partition['validation']))
print('Preflight passed. No optimizer updates yet.')


Output: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_16_review_protocol/runs/review_v1
Fingerprint: 2648555ff6bc6e62706e772358040b37fb1f726302a9b1e78e5c222a7252be70
primary | train IDs: 740 | validation IDs: 185
alternate | train IDs: 740 | validation IDs: 185
Preflight passed. No optimizer updates yet.


## 3. Запуск выбранного этапа

После прерывания снова **Restart Kernel → Run All** с теми же настройками.
Завершённые стадии проверяются и не переобучаются. Незавершённый блок до 200 updates повторится.
Каждый selectable checkpoint сохраняется: понадобится несколько гигабайт свободного места.

LR всегда рассчитан на 1700 updates, даже когда alternate/final останавливается раньше.
Это перенос правила числа updates, не воспроизведение старых 5 эпох MVP.


In [4]:
if DEVICE == 'cpu' and not ALLOW_CPU_TRAINING:
    raise RuntimeError('CPU-обучение отключено. Выбери MPS/CUDA или явно разреши CPU.')
result = run(context, phase=PHASE, allow_outer=ALLOW_OUTER_EVALUATION)
print('Complete:', result['complete'], '| Outer evaluated:', result['outer_evaluated'], '| Promoted:', result['promoted'])


final/B0_control/20260915: 200/800 | fixed rule; no selection | elapsed 00:03:06 | ETA 00:09:19
final/B0_control/20260915: 285/800 | fixed rule; no selection | elapsed 00:04:27 | ETA 00:08:02
final/B0_control/20260915: 400/800 | fixed rule; no selection | elapsed 00:06:14 | ETA 00:06:14
final/B0_control/20260915: 600/800 | fixed rule; no selection | elapsed 00:09:28 | ETA 00:03:09
final/B0_control/20260915: 800/800 | fixed rule; no selection | elapsed 00:12:43 | ETA 00:00:00
final/R1_resolution256/20260915: 200/800 | fixed rule; no selection | elapsed 00:04:01 | ETA 00:12:03
final/R1_resolution256/20260915: 285/800 | fixed rule; no selection | elapsed 00:05:37 | ETA 00:10:09
final/R1_resolution256/20260915: 400/800 | fixed rule; no selection | elapsed 00:07:48 | ETA 00:07:48
final/R1_resolution256/20260915: 600/800 | fixed rule; no selection | elapsed 00:11:37 | ETA 00:03:52
final/R1_resolution256/20260915: 800/800 | fixed rule; no selection | elapsed 00:15:19 | ETA 00:00:00


/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


Complete: True | Outer evaluated: True | Promoted: False


## 4. Результаты для разбора

Передай отчёт текущей фазы и одноимённый JSON. В confirmation смотри paired-seed разницы,
а не только максимальный mAP; исправленная ветвь сравнивается и с B0, и со старой color-ветвью.
На alternate сохраняются три draws, same-camera junk, confidence и автоматические masked diagnostics.
Эти маски **не подтверждены как plate-only**; по ним нельзя делать сильный вывод о зависимости от номеров.

Final создаёт experimental ONNX и frozen bundle. Экспорт submissions не требует train/calibration;
команды CPU/GPU и ограничения проверки скорости приведены в README. Ничего не копируется в MVP.


In [5]:
report = context['output'] / f'{PHASE.upper()}_RESULTS.md'
display(Markdown(report.read_text(encoding='utf-8')))
print('Detailed results:', context['output'] / f'{PHASE}.json')


# Review protocol: final

MVP unchanged; outer is historical development, not independent test.

| Recipe | Original reranked mAP@10 | F1 | TNR |
|---|---:|---:|---:|
| B0_control | 0.787521 | 0.724138 | 0.806452 |
| R1_resolution256 | 0.798955 | 0.777778 | 0.725806 |

NiVe/GeM/MixStyle are not added. Automatic masks are not verified plate-only masks.
No automatic promotion, upload, or next phase. See README for the next explicit step.


Detailed results: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_16_review_protocol/runs/review_v1/final.json
